# BST_CODE vs FOW consistency check

Check whether the BST filter `{RB, ERF, HR}` actually removes FOW=2 (multiple carriageway)
and FOW=4 (roundabout) roads, or whether those road types survive the filter.

In [ ]:
import geopandas as gpd
import pandas as pd
import os

PROJECT_DIR  = r"C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second year\Afstuderen\Project"
INCLUDE_BST  = {"RB", "ERF", "HR"}

# Load gemeente roads WITHOUT BST filter — full picture
wvk = gpd.read_file(os.path.join(PROJECT_DIR, "data", "processed", "wegvakken_rotterdam.gpkg"))
wvk["FOW_int"] = pd.to_numeric(wvk["FOW"], errors="coerce")

print(f"Total gemeente segments: {len(wvk):,}")

## 1. Cross-tabulation: BST_CODE × FOW

Shows how many segments of each BST_CODE have each FOW value.
Rows in the `RB / ERF / HR` bands that have FOW=2 or FOW=4 survive the BST filter.

In [ ]:
# Cross-tab: rows = BST_CODE, columns = FOW value
ct = pd.crosstab(wvk["BST_CODE"], wvk["FOW_int"], margins=True)
print(ct.to_string())

## 2. FOW=2 roads: which BST codes do they have?

If FOW=2 roads mostly have NRB/GRB/TRB codes they are filtered out.
If they have RB/HR codes they survive and the dual-carriageway detection can
use the BST-filtered dataset directly.

In [ ]:
fow2 = wvk[wvk["FOW_int"] == 2]
print(f"Total FOW=2 segments: {len(fow2):,}")
print()
print("BST_CODE distribution for FOW=2 roads:")
print(fow2["BST_CODE"].value_counts().to_string())
print()

# How many FOW=2 roads survive the BST filter?
fow2_in_bst = fow2[fow2["BST_CODE"].isin(INCLUDE_BST)]
print(f"FOW=2 roads that SURVIVE the BST filter: {len(fow2_in_bst):,} "
      f"({len(fow2_in_bst)/len(fow2)*100:.1f}%)")
if len(fow2_in_bst):
    print(fow2_in_bst["BST_CODE"].value_counts().to_string())

## 3. FOW=4 roads (roundabouts): which BST codes do they have?

In [ ]:
fow4 = wvk[wvk["FOW_int"] == 4]
print(f"Total FOW=4 segments: {len(fow4):,}")
print()
print("BST_CODE distribution for FOW=4 roads:")
print(fow4["BST_CODE"].value_counts().to_string())
print()

fow4_in_bst = fow4[fow4["BST_CODE"].isin(INCLUDE_BST)]
print(f"FOW=4 roads that SURVIVE the BST filter: {len(fow4_in_bst):,} "
      f"({len(fow4_in_bst)/len(fow4)*100:.1f}% of all FOW=4)")
if len(fow4_in_bst):
    print(fow4_in_bst["BST_CODE"].value_counts().to_string())

## 4. Summary: what is in the BST-filtered dataset?

FOW distribution of roads that actually survive the `{RB, ERF, HR}` filter.

In [ ]:
wvk_bst = wvk[wvk["BST_CODE"].isin(INCLUDE_BST)].copy()
print(f"Segments surviving BST filter: {len(wvk_bst):,}")
print()
print("FOW distribution inside BST-filtered dataset:")
print(wvk_bst["FOW_int"].value_counts().sort_index().to_string())